In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# Load and clean data
df = pd.read_csv("Dataset .csv")
df = df.dropna(subset=['Cuisines'])

# Extract primary cuisine
df['Cuisine_Primary'] = df['Cuisines'].apply(lambda x: x.split(',')[0].strip())

# Keep useful columns
df_rec = df[['Restaurant Name', 'Cuisines', 'Cuisine_Primary',
             'Average Cost for two', 'Price range',
             'Aggregate rating', 'Votes',
             'Has Table booking', 'Has Online delivery',
             'City', 'Locality']].copy()

# Create a combined text feature for TF-IDF
df_rec['combined_features'] = (
    df_rec['Cuisine_Primary'] + ' ' +
    df_rec['City'] + ' ' +
    df_rec['Price range'].astype(str) + ' ' +
    df_rec['Has Online delivery'] + ' ' +
    df_rec['Has Table booking']
)

print("✅ Data loaded!")
print("Shape:", df_rec.shape)
print("\nSample combined features:")
print(df_rec['combined_features'].head(3))

✅ Data loaded!
Shape: (9542, 12)

Sample combined features:
0          French Makati City 3 No Yes
1        Japanese Makati City 3 No Yes
2    Seafood Mandaluyong City 4 No Yes
Name: combined_features, dtype: str


In [2]:
# Step 2: Build the TF-IDF Matrix (the ML brain!)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_rec['combined_features'])

# Compute cosine similarity between all restaurants
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Reset index for easy lookup
df_rec = df_rec.reset_index(drop=True)

print("✅ ML Model built!")
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Cosine Similarity Matrix shape: {cosine_sim.shape}")
print(f"\nThis means every restaurant is compared to all {cosine_sim.shape[0]} others!")

✅ ML Model built!
TF-IDF Matrix shape: (9542, 286)
Cosine Similarity Matrix shape: (9542, 9542)

This means every restaurant is compared to all 9542 others!


In [3]:
# Step 3: Build the Recommendation Function
def recommend_restaurants(cuisine=None, city=None, 
                          price_range=None, min_rating=3.0, 
                          top_n=5):
    
    # Start with all restaurants
    filtered = df_rec.copy()
    
    # Apply user filters
    if cuisine:
        filtered = filtered[
            filtered['Cuisine_Primary'].str.contains(cuisine, case=False, na=False)
        ]
    if city:
        filtered = filtered[
            filtered['City'].str.contains(city, case=False, na=False)
        ]
    if price_range:
        filtered = filtered[filtered['Price range'] == price_range]
    
    filtered = filtered[filtered['Aggregate rating'] >= min_rating]
    
    if filtered.empty:
        print("❌ No restaurants found! Try relaxing your filters.")
        return None
    
    # Get cosine similarity scores for filtered restaurants
    filtered_indices = filtered.index.tolist()
    sim_scores = cosine_sim[filtered_indices].mean(axis=0)
    
    # Add similarity score and sort
    filtered = filtered.copy()
    filtered['similarity_score'] = sim_scores[filtered_indices]
    filtered['combined_score'] = (
        filtered['similarity_score'] * 0.4 +
        filtered['Aggregate rating'] / 5 * 0.4 +
        np.log1p(filtered['Votes']) / 10 * 0.2
    )
    
    # Return top N
    results = filtered.nlargest(top_n, 'combined_score')[
        ['Restaurant Name', 'Cuisine_Primary', 'City', 
         'Price range', 'Aggregate rating', 'Votes',
         'Has Online delivery', 'Has Table booking']
    ]
    return results

# Test it!
print("🍽️ TEST 1: Italian restaurants with rating > 3.5")
print("=" * 55)
result1 = recommend_restaurants(cuisine='Italian', min_rating=3.5)
print(result1.to_string(index=False))

print("\n🍽️ TEST 2: Cheap eats (price range 1) in any city")
print("=" * 55)
result2 = recommend_restaurants(price_range=1, min_rating=3.0)
print(result2.to_string(index=False))

🍽️ TEST 1: Italian restaurants with rating > 3.5
       Restaurant Name Cuisine_Primary      City  Price range  Aggregate rating  Votes Has Online delivery Has Table booking
             Big Chill         Italian New Delhi            3               4.5   4986                  No                No
             Big Chill         Italian New Delhi            3               4.6   1569                  No                No
The Flying Saucer Cafe         Italian New Delhi            3               4.3   3002                  No               Yes
             Big Chill         Italian New Delhi            3               4.4   2777                  No                No
             Big Chill         Italian New Delhi            3               4.4   1521                  No                No

🍽️ TEST 2: Cheap eats (price range 1) in any city
     Restaurant Name Cuisine_Primary      City  Price range  Aggregate rating  Votes Has Online delivery Has Table booking
  Naturals Ice Cream       

In [4]:
# Export all recommendation combinations for the dashboard
import json

cuisines_to_export = [
    'North Indian', 'Italian', 'Chinese', 'Japanese',
    'Fast Food', 'Cafe', 'Bakery', 'Seafood',
    'Street Food', 'Ice Cream', 'South Indian', 'Continental'
]

price_ranges = [1, 2, 3, 4]
min_ratings = [3.0, 3.5, 4.0, 4.5]

all_data = {}

# Generate all cuisine combinations
for cuisine in cuisines_to_export:
    for rating in min_ratings:
        key = f"{cuisine}_{rating}"
        result = recommend_restaurants(cuisine=cuisine, min_rating=rating, top_n=6)
        if result is not None and len(result) > 0:
            all_data[key] = []
            for _, row in result.iterrows():
                all_data[key].append({
                    "name": str(row['Restaurant Name']),
                    "cuisine": str(row['Cuisine_Primary']),
                    "city": str(row['City']),
                    "price": int(row['Price range']),
                    "rating": float(row['Aggregate rating']),
                    "votes": int(row['Votes']),
                    "delivery": str(row['Has Online delivery']),
                    "booking": str(row['Has Table booking'])
                })

# Generate all price range combinations
for price in price_ranges:
    for rating in min_ratings:
        key = f"price_{price}_{rating}"
        result = recommend_restaurants(price_range=price, min_rating=rating, top_n=6)
        if result is not None and len(result) > 0:
            all_data[key] = []
            for _, row in result.iterrows():
                all_data[key].append({
                    "name": str(row['Restaurant Name']),
                    "cuisine": str(row['Cuisine_Primary']),
                    "city": str(row['City']),
                    "price": int(row['Price range']),
                    "rating": float(row['Aggregate rating']),
                    "votes": int(row['Votes']),
                    "delivery": str(row['Has Online delivery']),
                    "booking": str(row['Has Table booking'])
                })

# Dataset stats for charts
cuisine_counts = df_rec['Cuisine_Primary'].value_counts().head(8)
city_counts = df_rec['City'].value_counts().head(6)

print(f"✅ Generated {len(all_data)} recommendation combinations!")
print("Ready to build dashboard!")

❌ No restaurants found! Try relaxing your filters.
✅ Generated 63 recommendation combinations!
Ready to build dashboard!


In [6]:
# Build dashboard - Part 1: Setup
import json

cuisine_counts = df_rec['Cuisine_Primary'].value_counts().head(8)
city_counts = df_rec['City'].value_counts().head(6)

cuisine_labels = json.dumps(cuisine_counts.index.tolist())
cuisine_values = json.dumps(cuisine_counts.values.tolist())
city_labels = json.dumps(city_counts.index.tolist())
city_values = json.dumps(city_counts.values.tolist())
all_data_json = json.dumps(all_data)

unique_cuisines = len(df_rec['Cuisine_Primary'].unique())
unique_cities = len(df_rec['City'].unique())

print("✅ Part 1 ready!")
print(f"Cuisines: {unique_cuisines}, Cities: {unique_cities}")

✅ Part 1 ready!
Cuisines: 119, Cities: 140


In [7]:
# Build dashboard - Part 2: Generate HTML
html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Restaurant Recommender | Cognifyz Internship</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<style>
* { margin:0; padding:0; box-sizing:border-box; }
body { font-family:'Segoe UI',sans-serif; background:#0f172a; color:#e2e8f0; min-height:100vh; }
.header { background:linear-gradient(135deg,#f97316,#ec4899,#8b5cf6); padding:50px 40px; text-align:center; }
.header h1 { font-size:2.8rem; font-weight:800; color:white; margin-bottom:10px; }
.header p { color:rgba(255,255,255,0.85); font-size:1.1rem; }
.stats-row { display:flex; gap:20px; padding:30px 40px; justify-content:center; flex-wrap:wrap; }
.stat-card { background:linear-gradient(135deg,#1e293b,#334155); border-radius:16px; padding:24px 32px; text-align:center; border:1px solid #475569; min-width:160px; }
.stat-card .value { font-size:2rem; font-weight:800; color:#fb923c; }
.stat-card .label { font-size:0.85rem; color:#94a3b8; margin-top:6px; }
.ml-info { text-align:center; padding:0 40px 20px; }
.ml-badge { background:linear-gradient(135deg,#8b5cf6,#ec4899); color:white; padding:5px 14px; border-radius:999px; font-size:0.8rem; font-weight:600; display:inline-block; margin:4px; }
.section { padding:0 40px 40px; }
.section-title { font-size:1.3rem; font-weight:700; color:#fb923c; margin-bottom:20px; padding-bottom:10px; border-bottom:2px solid #334155; }
.search-box { background:#1e293b; border-radius:20px; padding:32px; border:1px solid #334155; margin-bottom:30px; }
.search-title { color:#f1f5f9; font-size:1.2rem; font-weight:700; margin-bottom:24px; }
.filters { display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:16px; margin-bottom:24px; }
.filter-group label { display:block; font-size:0.78rem; color:#94a3b8; margin-bottom:8px; text-transform:uppercase; letter-spacing:0.08em; font-weight:600; }
.filter-group select { width:100%; padding:12px 16px; background:#0f172a; border:1px solid #475569; border-radius:10px; color:#e2e8f0; font-size:0.95rem; cursor:pointer; }
.filter-group select:focus { outline:none; border-color:#f97316; }
.btn { background:linear-gradient(135deg,#f97316,#ec4899); color:white; border:none; padding:14px 32px; border-radius:10px; font-size:1rem; font-weight:700; cursor:pointer; width:100%; }
.btn:hover { opacity:0.9; }
.results-header { color:#94a3b8; font-size:0.9rem; margin-bottom:16px; min-height:24px; }
.cards-grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(300px,1fr)); gap:20px; }
.rest-card { background:#1e293b; border-radius:16px; padding:22px; border:1px solid #334155; transition:all 0.25s; }
.rest-card:hover { transform:translateY(-5px); border-color:#f97316; box-shadow:0 8px 30px rgba(249,115,22,0.15); }
.card-top { display:flex; justify-content:space-between; align-items:flex-start; margin-bottom:8px; }
.rest-name { font-size:1.05rem; font-weight:700; color:#f1f5f9; flex:1; margin-right:10px; }
.rest-rating { font-size:1.2rem; font-weight:800; color:#fbbf24; white-space:nowrap; }
.rest-cuisine { color:#fb923c; font-size:0.88rem; margin-bottom:14px; font-weight:500; }
.rest-details { display:flex; flex-wrap:wrap; gap:8px; }
.badge { background:#0f172a; border-radius:999px; padding:4px 12px; font-size:0.78rem; color:#94a3b8; border:1px solid #334155; }
.badge.green { color:#34d399; border-color:#34d399; }
.badge.blue { color:#60a5fa; border-color:#60a5fa; }
.badge.orange { color:#fb923c; border-color:#fb923c; }
.votes { color:#64748b; font-size:0.8rem; margin-top:10px; }
.no-results { text-align:center; padding:40px; color:#94a3b8; font-size:1rem; grid-column:1/-1; }
.charts-grid { display:grid; grid-template-columns:1fr 1fr; gap:24px; }
.chart-card { background:#1e293b; border-radius:16px; padding:24px; border:1px solid #334155; }
.chart-card h3 { font-size:0.9rem; font-weight:700; color:#fb923c; margin-bottom:16px; text-transform:uppercase; letter-spacing:0.06em; }
canvas { max-height:260px; }
.footer { text-align:center; padding:30px; color:#475569; font-size:0.85rem; border-top:1px solid #1e293b; margin-top:20px; }
</style>
</head>
<body>
<div class="header">
  <h1>🍽️ Restaurant Recommender</h1>
  <p>Powered by TF-IDF Vectorization & Cosine Similarity | Cognifyz Internship Task 2</p>
</div>
<div class="stats-row">
  <div class="stat-card"><div class="value">9,542</div><div class="label">Restaurants</div></div>
  <div class="stat-card"><div class="value" style="color:#34d399">""" + str(unique_cuisines) + """</div><div class="label">Cuisine Types</div></div>
  <div class="stat-card"><div class="value" style="color:#60a5fa">""" + str(unique_cities) + """</div><div class="label">Cities</div></div>
  <div class="stat-card"><div class="value" style="color:#f472b6">91M+</div><div class="label">ML Comparisons</div></div>
  <div class="stat-card"><div class="value" style="color:#a78bfa">63</div><div class="label">Filter Combos</div></div>
</div>
<div class="ml-info">
  <span class="ml-badge">✅ TF-IDF Vectorization</span>
  <span class="ml-badge">✅ Cosine Similarity</span>
  <span class="ml-badge">✅ Content-Based Filtering</span>
  <span class="ml-badge">✅ Multi-factor Scoring</span>
</div>
<div class="section">
  <div class="search-box">
    <div class="search-title">🔍 Find Your Perfect Restaurant</div>
    <div class="filters">
      <div class="filter-group">
        <label>Cuisine Type</label>
        <select id="cuisine">
          <option value="">Any Cuisine</option>
          <option>North Indian</option>
          <option>Italian</option>
          <option>Chinese</option>
          <option>Japanese</option>
          <option>Fast Food</option>
          <option>Cafe</option>
          <option>Bakery</option>
          <option>Seafood</option>
          <option>Street Food</option>
          <option>Ice Cream</option>
          <option>South Indian</option>
          <option>Continental</option>
        </select>
      </div>
      <div class="filter-group">
        <label>Price Range</label>
        <select id="price">
          <option value="">Any Price</option>
          <option value="1">💰 Budget</option>
          <option value="2">💰💰 Moderate</option>
          <option value="3">💰💰💰 Expensive</option>
          <option value="4">💰💰💰💰 Fine Dining</option>
        </select>
      </div>
      <div class="filter-group">
        <label>Minimum Rating</label>
        <select id="rating">
          <option value="3.0">⭐ 3.0+</option>
          <option value="3.5">⭐ 3.5+</option>
          <option value="4.0" selected>⭐ 4.0+</option>
          <option value="4.5">⭐ 4.5+</option>
        </select>
      </div>
    </div>
    <button class="btn" onclick="getRecommendations()">🚀 Get My Recommendations</button>
  </div>
  <div class="results-header" id="resultsHeader"></div>
  <div class="cards-grid" id="resultsGrid"></div>
</div>
<div class="section">
  <div class="section-title">📊 Dataset Insights</div>
  <div class="charts-grid">
    <div class="chart-card"><h3>🍜 Top Cuisines</h3><canvas id="cuisineChart"></canvas></div>
    <div class="chart-card"><h3>🏙️ Restaurants by City</h3><canvas id="cityChart"></canvas></div>
  </div>
</div>
<div class="footer">Built by Harini | Cognifyz Internship Task 2 | Restaurant Recommendation System using ML</div>
<script>
const allData = """ + all_data_json + """;
function getPriceEmoji(p) { return '💰'.repeat(p); }
function getRecommendations() {
  const cuisine = document.getElementById('cuisine').value;
  const price = document.getElementById('price').value;
  const rating = document.getElementById('rating').value;
  const grid = document.getElementById('resultsGrid');
  const header = document.getElementById('resultsHeader');
  let results = null;
  if (cuisine) {
    const key = cuisine + '_' + rating;
    if (allData[key]) results = allData[key];
  }
  if (!results && price) {
    const key = 'price_' + price + '_' + rating;
    if (allData[key]) results = allData[key];
  }
  if (!results && cuisine) {
    const relaxed = ['3.5', '3.0'];
    for (let r of relaxed) {
      const key = cuisine + '_' + r;
      if (allData[key]) { results = allData[key]; break; }
    }
  }
  if (!results || results.length === 0) {
    header.innerHTML = '';
    grid.innerHTML = '<div class="no-results">😕 No restaurants found. Try different filters!</div>';
    return;
  }
  header.innerHTML = 'Showing <strong style="color:#fb923c">' + results.length + ' recommendations</strong> based on your preferences';
  grid.innerHTML = results.map(r => `
    <div class="rest-card">
      <div class="card-top">
        <div class="rest-name">${r.name}</div>
        <div class="rest-rating">⭐ ${r.rating}</div>
      </div>
      <div class="rest-cuisine">${r.cuisine}</div>
      <div class="rest-details">
        <span class="badge orange">📍 ${r.city}</span>
        <span class="badge">${getPriceEmoji(r.price)}</span>
        ${r.delivery === 'Yes' ? '<span class="badge green">🛵 Delivery</span>' : ''}
        ${r.booking === 'Yes' ? '<span class="badge blue">📅 Booking</span>' : ''}
      </div>
      <div class="votes">👥 ${r.votes.toLocaleString()} votes</div>
    </div>
  `).join('');
}
new Chart(document.getElementById('cuisineChart'), {
  type: 'bar',
  data: {
    labels: """ + cuisine_labels + """,
    datasets: [{ data: """ + cuisine_values + """, backgroundColor: ['#f97316','#ec4899','#8b5cf6','#60a5fa','#34d399','#fbbf24','#fb923c','#e879f9'], borderRadius: 6 }]
  },
  options: { plugins: { legend: { display:false } }, scales: { x: { ticks: { color:'#94a3b8', font:{size:10} }, grid: { display:false } }, y: { ticks: { color:'#94a3b8' }, grid: { color:'#334155' } } } }
});
new Chart(document.getElementById('cityChart'), {
  type: 'doughnut',
  data: {
    labels: """ + city_labels + """,
    datasets: [{ data: """ + city_values + """, backgroundColor: ['#f97316','#ec4899','#8b5cf6','#60a5fa','#34d399','#fbbf24'], borderWidth:0 }]
  },
  options: { plugins: { legend: { labels: { color:'#e2e8f0', font:{size:11} } } } }
});
getRecommendations();
</script>
</body>
</html>"""

with open("recommendation_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)

print("✅ Dashboard created!")
print("Open 'recommendation_dashboard.html' in your browser!")

✅ Dashboard created!
Open 'recommendation_dashboard.html' in your browser!
